In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth

In [ ]:
%%capture
# Install latest transformers for Gemma 3N
!pip install --no-deps --upgrade transformers # Only for Gemma 3N
!pip install --no-deps --upgrade timm # Only for Gemma 3N

In [ ]:
from unsloth import FastModel
import torch

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3n-E4B-it-unsloth-bnb-4bit",
    "unsloth/gemma-3n-E2B-it-unsloth-bnb-4bit",
    # Pretrained models
    "unsloth/gemma-3n-E4B-unsloth-bnb-4bit",
    "unsloth/gemma-3n-E2B-unsloth-bnb-4bit",

    # Other Gemma 3 quants
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3n-E2B-it", # Or "unsloth/gemma-3n-E4B-it"
    dtype = None, # None for auto detection
    max_seq_length = 4096, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_nlp_modules       = True,  # Should leave on always!

    r = 16,           # Larger = higher accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

In [ ]:
# Load a structured chat prompt format into your tokenizer
from unsloth.chat_templates import get_chat_template as attach_template

def format_tokenizer_for_chat(tok, template="gemma-3"):
    """Adds a conversational structure to the tokenizer for chat-style input."""
    return attach_template(tok, chat_template=template)

tokenizer = format_tokenizer_for_chat(tokenizer)

In [ ]:

from datasets import load_dataset, Dataset
import random, json

def preview_dataset(name, subset, max_rows=None):
    """Utility to load and preview a HuggingFace dataset subset."""
    data = load_dataset(name, split=subset)
    if max_rows:
        data = data.select(range(max_rows))
    print(f" Loaded '{name}' with {len(data)} entries.")
    return data

# Psychology dataset → distinguishing empathy vs. dismissive responses
psych_data = preview_dataset("jkhedri/psychology-dataset", "train", max_rows=3000)

# Coaching dataset → distinguishing questioning vs. directive leadership styles
coach_data = preview_dataset("drublackberry/hbr-coaching-real-leaders", "train")


In [ ]:
# %%
# Parsing datasets and preparing for DPO conversion
from datasets import Dataset
import gc

print("Step 1: Parsing both datasets...")

# Function to process the psychology dataset
def process_psychology_data(psych_source, max_samples=3000):
    """Process psychology dataset and prepare for DPO"""

    print(f"Psychology dataset: {len(psych_source)} total rows, using {max_samples}")

    parsed_samples = []

    # Coaching prompts to enrich psychology responses
    coaching_prompts = [
        "What feels most important to you as you think about this?",
        "What would you like to focus on moving forward?",
        "How would you like to approach this situation?",
        "What support would be most helpful for you right now?",
        "What small step could you take today?",
        "What matters most to you in this situation?",
        "How do you want to move forward with this?",
        "What would success look like for you here?"
    ]

    for i in range(min(max_samples, len(psych_source))):
        if i % 500 == 0:
            print(f"Processing psychology: {i}/{max_samples}")
            gc.collect()

        row = psych_source[i]

        # Add enrichment to good response
        prompt_addition = coaching_prompts[i % len(coaching_prompts)]
        enriched_response = f"{row['response_j']}\n\n{prompt_addition}"

        parsed_samples.append({
            "prompt": row['question'],
            "chosen": enriched_response,
            "rejected": row['response_k']
        })

    return parsed_samples

# Function to process coaching dataset
def process_coaching_data(coach_source):
    """Extract useful coaching pairs from dataset"""

    print(f"Coaching dataset: {len(coach_source)} rows")

    coaching_pairs = []

    for idx, row in enumerate(coach_source):
        messages = row['messages']

        for i in range(len(messages) - 1):
            current = messages[i]
            next_msg = messages[i + 1]

            # Ensure valid user → assistant exchange
            if (
                current.get('role') == 'user'
                and next_msg.get('role') == 'assistant'
                and len(current.get('content', '')) > 50
                and len(next_msg.get('content', '')) > 30
                and '?' in next_msg.get('content', '')
            ):
                client_text = current['content'].strip()
                coach_text = next_msg['content'].strip()

                # Artificial "bad coaching" response
                poor_response = f"You should just {client_text.lower().split()[0]} differently. Stop overthinking and take action immediately."

                coaching_pairs.append({
                    "prompt": client_text,
                    "chosen": coach_text,
                    "rejected": poor_response
                })

    print(f"Extracted {len(coaching_pairs)} coaching pairs")
    return coaching_pairs

# Run parsing
psychology_samples = process_psychology_data(psych_data, max_samples=3000)
coaching_samples = process_coaching_data(coach_data)

# Combine
all_samples = psychology_samples + coaching_samples
print(f"Total parsed examples: {len(all_samples)}")

# Step 2: Convert to DPO format
print("Step 2: Converting to DPO format...")

def convert_to_dpo(samples):
    """Convert parsed data into DPO-compatible format"""

    formatted = []

    for i, ex in enumerate(samples):
        if i % 500 == 0:
            print(f"Converting: {i}/{len(samples)}")
            gc.collect()

        formatted.append({
            "prompt": ex['prompt'],
            "chosen": ex['chosen'],
            "rejected": ex['rejected']
        })

    return formatted

# Convert and finalize
dpo_data = convert_to_dpo(all_samples)
final_dataset = Dataset.from_list(dpo_data)

# Cleanup memory
del psych_data, coach_data, psychology_samples, coaching_samples, all_samples, dpo_data
gc.collect()

print(f"Final DPO dataset ready: {len(final_dataset)} examples")


In [ ]:
# data_utils.py
import os
import gc
from datasets import load_dataset, Dataset


def load_and_validate_datasets():
    """
    Load psychology and coaching datasets, then clean and validate them
    for ultra-safe training.

    Returns:
        ultra_safe_dataset: HuggingFace Dataset, cleaned and ready for training
    """
    # Force single-threaded processing
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"

    print("[INFO] Loading psychology dataset...")
    psych_dataset = load_dataset("jkhedri/psychology-dataset", split="train[:3000]")

    print("[INFO] Loading executive coaching dataset...")
    coaching_dataset = load_dataset("drublackberry/hbr-coaching-real-leaders", split="train")

    print(f"[INFO] Psychology dataset: {len(psych_dataset)} examples")
    print(f"[INFO] Coaching dataset: {len(coaching_dataset)} examples")

    # Merge datasets if needed (example: append coaching to psychology)
    combined_dataset = psych_dataset + coaching_dataset

    print("[INFO] Validating and cleaning combined dataset...")

    clean_examples = []
    for i, example in enumerate(combined_dataset):
        if (example and isinstance(example, dict) and
            example.get("prompt") and example.get("chosen") and example.get("rejected") and
            isinstance(example["prompt"], str) and isinstance(example["chosen"], str) and isinstance(example["rejected"], str) and
            len(example["prompt"].strip()) > 5 and len(example["chosen"].strip()) > 5 and len(example["rejected"].strip()) > 5
           ):
            clean_examples.append({
                "prompt": example["prompt"].strip(),
                "chosen": example["chosen"].strip(),
                "rejected": example["rejected"].strip()
            })
        else:
            print(f"[WARN] Skipping invalid example {i}")

    ultra_safe_dataset = Dataset.from_list(clean_examples)

    # Clear memory
    del combined_dataset, clean_examples, psych_dataset, coaching_dataset
    gc.collect()

    print(f"[INFO] Ultra-safe dataset ready: {len(ultra_safe_dataset)} examples")
    return ultra_safe_dataset


In [ ]:
from datasets import Dataset

# Example: create a Dataset from a list of dictionaries
# Replace this with your real dataset
my_data_list = [
    {"prompt": "Hello, how are you?", "chosen": "I'm good, thanks!"},
    {"prompt": "What's AI?", "chosen": "AI is artificial intelligence."},
]

# Convert to Hugging Face Dataset
cleaned = Dataset.from_list(my_data_list)


In [ ]:
# Ultra-safe DPO training: single-threaded, stable configuration
from trl import DPOTrainer, DPOConfig

print("Starting ultra-safe DPO training...")

try:
    trainer = DPOTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=ultra_safe_dataset,  # Use cleaned dataset
        args=DPOConfig(
            output_dir="./wellness-coach-safe",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=1,
            max_steps=100,
            learning_rate=1e-6,
            warmup_steps=10,
            logging_steps=10,
            save_steps=25,
            optim="adamw_8bit",
            gradient_checkpointing=True,
            fp16=True,

            # Single-threaded for stability
            dataloader_num_workers=0,
            preprocessing_num_workers=1,
            dataloader_persistent_workers=False,
            dataloader_pin_memory=False,

            # DPO-specific parameters
            beta=0.1,
            loss_type="sigmoid",

            # Other safety settings
            seed=3407,
            report_to="none",
            disable_tqdm=False,
            remove_unused_columns=True,
        ),
    )

    print("Trainer configured successfully.")
    print(f"Training on {len(ultra_safe_dataset)} examples.")

    # Start training
    stats = trainer.train()

    print("Training completed successfully.")
    print(f"Training runtime: {round(stats.metrics['train_runtime']/60, 2)} minutes.")

    # Save trained model and tokenizer
    model.save_pretrained("wellness-coach-trained")
    tokenizer.save_pretrained("wellness-coach-trained")
    print("Model saved successfully.")

except Exception as e:
    print(f"DPO training failed: {e}")
    print("Attempting fallback SFT training...")


In [ ]:
# %%
# ULTRA-SAFE TRAINING PIPELINE: DPO first, SFT fallback
from trl import DPOTrainer, DPOConfig, SFTTrainer, SFTConfig
from datasets import Dataset
from unsloth.chat_templates import standardize_data_formats

# -------------------------------
# STEP 0: Define your dataset
# Replace with your preprocessed dataset variable if different
try:
    data = cleaned  # Your preprocessed dataset variable
except NameError:
    raise NameError("Dataset variable 'cleaned' not found. Assign your preprocessed dataset to 'cleaned'.")

# -------------------------------
# STEP 1: ULTRA-SAFE DPO TRAINING
print("Starting ultra-safe DPO training...")

try:
    dpo_trainer = DPOTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=data,
        args=DPOConfig(
            output_dir="./wellness-coach-safe",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=1,
            max_steps=100,
            learning_rate=1e-6,
            warmup_steps=10,
            logging_steps=10,
            save_steps=25,
            optim="adamw_8bit",
            gradient_checkpointing=True,
            fp16=True,
            dataloader_num_workers=0,
            preprocessing_num_workers=1,
            dataloader_persistent_workers=False,
            dataloader_pin_memory=False,
            beta=0.1,
            loss_type="sigmoid",
            seed=3407,
            report_to="none",
            disable_tqdm=False,
            remove_unused_columns=True,
        ),
    )

    print("Ultra-safe DPO trainer configured.")
    print(f"Training {len(data)} examples.")

    trainer_stats = dpo_trainer.train()

    print("DPO training completed successfully.")
    print(f"Training time: {round(trainer_stats.metrics['train_runtime']/60, 2)} minutes")

    model.save_pretrained("wellness-coach-trained")
    tokenizer.save_pretrained("wellness-coach-trained")
    print("Model saved successfully.")

except Exception as e:
    print(f"DPO training failed: {e}")
    print("Attempting fallback SFT training...")

    # -------------------------------
    # STEP 2: SFT FALLBACK TRAINING
    def convert_dpo_to_sft(dpo_dataset):
        sft_examples = []
        for example in dpo_dataset:
            conversation = [
                {"role": "user", "content": example['prompt']},
                {"role": "assistant", "content": example['chosen']}
            ]
            sft_examples.append({"conversations": conversation})
        return sft_examples

    sft_data = convert_dpo_to_sft(data)
    sft_dataset = Dataset.from_list(sft_data)

    sft_dataset = standardize_data_formats(sft_dataset)

    def formatting_prompts_func(examples):
        convos = examples["conversations"]
        texts = [
            tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False).removeprefix('<bos>')
            for convo in convos
        ]
        return {"text": texts}

    sft_dataset = sft_dataset.map(formatting_prompts_func, batched=True)

    sft_trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=sft_dataset,
        args=SFTConfig(
            dataset_text_field="text",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=2,
            warmup_steps=10,
            max_steps=100,
            learning_rate=2e-6,
            logging_steps=10,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=3407,
            report_to="none",
            output_dir="./wellness-coach-sft",
            dataloader_num_workers=0,
            remove_unused_columns=True,
            gradient_checkpointing=True,
            fp16=True,
        ),
    )

    print("SFT trainer configured as fallback.")
    print("SFT will train on good responses only.")

    # Uncomment to start fallback training
    # sft_trainer.train()


In [ ]:
"""### Train the Enhanced Wellness Coach"""

print(" Starting Enhanced Wellness Coach DPO Training...")
print("Learning: Empathy vs Dismissive + Questions vs Directive advice")
print("  This will take some time on T4...")

trainer_stats = sft_trainer.train()

In [ ]:
# %%
# TEST: Check if the model actually learned anything
def test_wellness_coach():
    """Test the trained wellness coach model on example prompts"""

    test_prompts = [
        "I've been feeling really anxious about my job performance lately.",
        "I can't seem to stick to my exercise routine.",
        "I feel overwhelmed with everything in my life right now.",
        "I'm torn between a safe career move and a risky opportunity."
    ]

    print("Testing trained wellness coach model...")

    for idx, prompt in enumerate(test_prompts):
        messages = [{
            "role": "user",
            "content": [{"type": "text", "text": prompt}]
        }]

        # Prepare model inputs
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            tokenize=True,
            return_dict=True,
        ).to("cuda")

        print(f"\n{'='*60}")
        print(f"Test {idx+1}: {prompt}")
        print("Response: ", end="")

        from transformers import TextStreamer

        _ = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            top_p=0.9,
            top_k=50,
            streamer=TextStreamer(tokenizer, skip_prompt=True),
        )

# Execute the test
test_wellness_coach()


In [ ]:
# %%
# ENHANCED MULTI-PROMPT TEST: Creative & consoling responses
def test_multiple_creative_responses():
    """Test multiple prompts and store enhanced responses"""

    test_prompts = [
        "I'm feeling burned out at work and it's affecting my relationships.",
        "I feel anxious and unsure about my career path.",
        "Lately, I can't find motivation to maintain my daily routines.",
        "I feel overwhelmed trying to balance personal and professional life."
    ]

    from transformers import TextStreamer

    results = []  # Store prompts and responses

    for i, prompt in enumerate(test_prompts):
        messages = [{
            "role": "user",
            "content": [{"type": "text", "text": prompt}]
        }]

        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            tokenize=True,
            return_dict=True,
        ).to("cuda")

        print(f"\n{'='*60}\nTest {i+1}: {prompt}\nEnhanced Response: ", end="")

        # Use a streamer to display in real-time while also capturing
        streamer = TextStreamer(tokenizer, skip_prompt=True)
        output = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.85,
            top_p=0.95,
            top_k=60,
            streamer=streamer,
        )

        # Store the prompt and response
        results.append({
            "prompt": prompt,
            "response": tokenizer.decode(output[0], skip_special_tokens=True)
        })

    return results

# Run the multi-prompt test
enhanced_responses = test_multiple_creative_responses()

# Example: Access the first response
print("\nFirst stored response:\n", enhanced_responses[0])
